# 📖 Novel-TUI — Servidor LLM Remoto (GPU T4 en Google Colab)

Este cuaderno ejecuta **KoboldCpp** con aceleración CUDA en GPU T4 y el modelo **L3-8B-Stheno-v3.2** (GGUF Q5_K_M sin censura).

### ⚡ Ventajas:
1. **Persistencia en Google Drive:** El modelo se descarga una sola vez en tu Drive (`/NovelTUI_Models/`) y en los próximos arranques inicia en 5 segundos.
2. **Túnel Cloudflare Gratuito:** Genera una URL pública `https://*.trycloudflare.com/v1` compatible con la API de OpenAI para Novel-TUI.

In [ ]:
#@title 🚀 Iniciar Servidor KoboldCpp con GPU y Google Drive
import os
import subprocess
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Rutas y URLs
DRIVE_DIR = '/content/drive/MyDrive/NovelTUI_Models'
MODEL_PATH = os.path.join(DRIVE_DIR, 'L3-8B-Stheno-v3.2-Q5_K_M.gguf')
MODEL_URL = 'https://huggingface.co/bartowski/L3-8B-Stheno-v3.2-GGUF/resolve/main/L3-8B-Stheno-v3.2-Q5_K_M.gguf'
KOBOLD_URL = 'https://github.com/LostRuins/koboldcpp/releases/download/v1.78/koboldcpp-linux-x64-cuda1200'

os.makedirs('/content/novel-llm', exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
os.chdir('/content/novel-llm')

# 3. Descargar KoboldCpp si no existe
if not os.path.exists('/content/novel-llm/koboldcpp'):
    print('📥 Descargando KoboldCpp CUDA...')
    subprocess.run(['wget', '-q', '-c', KOBOLD_URL, '-O', 'koboldcpp'], check=True)
    subprocess.run(['chmod', '+x', 'koboldcpp'], check=True)

# 4. Verificar si el modelo ya está en Google Drive
if os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 1000000000:
    print('⚡ Modelo encontrado en Google Drive. Creando enlace...')
    if os.path.exists('model.gguf'):
        os.remove('model.gguf')
    os.symlink(MODEL_PATH, 'model.gguf')
else:
    print('📥 Descargando L3-8B-Stheno a Google Drive (5.7 GB - demora ~1 min en Colab)...')
    subprocess.run(['wget', '-c', MODEL_URL, '-O', MODEL_PATH], check=True)
    if os.path.exists('model.gguf'):
        os.remove('model.gguf')
    os.symlink(MODEL_PATH, 'model.gguf')

# 5. Iniciar KoboldCpp con todas las capas en GPU y Cloudflare tunnel
print('🚀 Iniciando KoboldCpp con GPU y Túnel Cloudflare...')
cmd = [
    './koboldcpp',
    '--model', 'model.gguf',
    '--usecublas',
    '--gpulayers', '33',
    '--contextsize', '8192',
    '--usecloudflare',
    '--skiplauncher'
]
subprocess.run(cmd)
